In [1]:
# third-party imports
import pandas as pd 

In [2]:
df = pd.read_csv("OUTPUTS/dataframe_02_3.csv", 
                 dtype={
                     "Russian Translation": "string",
                     "English Translation": "string"}
                )
df.head(2)

/var/folders/ps/86qydyw53xj72p_g1g25hfx00000gn/T/ipykernel_78889/2581440014.py:1: DtypeWarning: Columns (20) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv("OUTPUTS/dataframe_02_3.csv",


,Unnamed: 0,File,Text Title,Language,Sentence ID,Token ID,Form,Lemma,Lemma_norm,POS,...,Presentation After,Russian Translation,English Translation,Type,century,exact,lang,source,place,region
0,0,mst,Mstislav’s letter,orv,189407,2157773,Се,се,се,I-,...,,"вот, это","behold, here is",OR,12,1130,OR,NaN,Novgorod,East Slavic
1,1,mst,Mstislav’s letter,orv,189407,2157774,азъ,азъ,аз_,Pp,...,,я,I,OR,12,1130,OR,NaN,Novgorod,East Slavic


In [3]:
def count_column_value_counts_verbs(df, column, column_value, column2, column_value2):
    """
    Desc:
        - Filter df so that only columns with 
        df[column] == column_value AND  df[column2] == column_value2 remain
        - Group by "Text Title" and "File", then count these combinations

    Returns: 
        -  For each combination of ("Text Title" and "File"): sum 
    """
    # 1. Filter: beide Vergleiche in Klammern, exakt POS == "V-"
    df_subset_verbs = df.loc[
        (df[column] == column_value) &
        (df[column2] == column_value2),
        :
    ]

    # 2. Gruppieren und zählen
    combi_counts_verbs = (
        df_subset_verbs
        .groupby(["Text Title", "File"])
        .size()
        .reset_index(name="count_verbs")
    )

    # 3. Ausgabe
    print(f"\nSUMME aller Tokens in dieser Teilmenge ({column} == '{column_value}' und {column2} == '{column_value2}'):")
    print(combi_counts_verbs["count_verbs"].sum())

    return combi_counts_verbs

In [4]:
def x_col_times_y_col(df, col_file, col_cent):
    """
    Return table with combinations of:
        col "File" * col "century"
    """
    # 1. Group and count combinations
    counts_ = df.groupby([col_file, col_cent]).size()

    # 2. If no value: fill with "0":
    table = counts_.unstack(fill_value=0).sort_index(axis=1)
    return table

# Create mask for the verbs  
mask_verbs = df["POS"].fillna("").str.startswith("V-")

# Apply filter mask
verbs_only = df[mask_verbs]

# Apply function
freq_file_century = x_col_times_y_col(verbs_only, "File", "century")
print(freq_file_century)

century              11     12    13   14    15    16    17
File                                                       
afnik                 0      0     0    0   804     0     0
avv                   0      0     0    0     0     0  4511
birchbark             8    185    90   61    11     0     0
const                 0      0     0    0  1684     0     0
domo                  0      0     0    0     0  3589     0
drac                  0      0     0    0   546     0     0
dux-grjaz             0      0     0    0     0    34     0
kiev-hyp              0     95     0    0     0     0     0
lav                   0  11477     0    0     0     0     0
luk-koloc             0      0     0    0   152     0     0
mst                   0     23     0    0     0     0     0
mstislav-col          0     34     0    0     0     0     0
nov-list              0      0     0    0    38     0     0
nov-marg              0      0    15    0     0     0     0
nov-sin               0      0  2706    

In [5]:
def groupby_col_row_size_table_optional_filter(
    df,
    col,
    row,
    filter_col=None,
    filter_val=None
):
    """
    Group df by col1 and col2 and create Pivot-Table:
        col -> returns  Table_Col, 
        row -> returns Table_Row
    
    Counts the frequencies of col1-col2 combinations
    for special filter_val in filter_col 
    (e.g. filter_col "POS" with containing filter_val "V-")
    """
    #  Flter None entries
    if filter_col and filter_val is not None:
        df = df[df[filter_col] == filter_val]

    # Create Pivot table
    counts = df.groupby([col, row]).size().unstack(fill_value=0)

    # Sort columns numerically 
    numeric = [c for c in counts.columns if str(c).isdigit()]
    others  = [c for c in counts.columns if c not in numeric]
    sorted_cols = sorted(numeric, key=lambda x: int(x)) + sorted(others)
    table = counts[sorted_cols]

    # 3) Create column "places"
    if 'place' in df.columns:
        exp = df.copy()
        # if list of "places" is empty:
        if exp['place'].apply(lambda x: isinstance(x, list)).any():
            exp = exp.explode('place')
        places = (
            exp.dropna(subset=['place'])
               .drop_duplicates(subset=[col, 'place'])
               .groupby(col)['place']
               .apply(lambda lst: sorted(lst.tolist()))
        )
        table['places'] = places

    return table

In [6]:
table_place_century = groupby_col_row_size_table_optional_filter(df, "place", "century")
print(table_place_century)

century              11     12     13    14     15     16     17  \
place                                                              
Byzantine_Empire      0      0      0     0   9258      0      0   
Kiev              25684   7568      0     0      0      0      0   
Latvia                0      0    171     0      0      0      0   
Moscow                0  23760      0  2399  22687  23880  24089   
Nizhny_Novgorod       0  56725      0     0      0      0      0   
NorthernRussia        0      0      0     0   2487      0   1835   
Novgorod            248   1487  18799   352    270      0      0   
Pskov                 0   3610     56     0   1245      0      0   
Smolensk              0      0   1421   344      0      0      0   
Staraja_Russa         0     42      0     0      0      0      0   
Tver                  0      0      0     0   6842      0      0   
Unknown               0     16      0     0      0      0      0   

century                       places  
place   

In [7]:
table = groupby_col_row_size_table_optional_filter(
    df,
    col='region',
    row='century',
    filter_col='POS',
    filter_val='V-'
)
print(table)

century             11     12    13   14    15    16    17  \
region                                                       
Byzantine_Empire     0      0     0    0  1684     0     0   
East Slavic       5145  17992  3087  455  5730  3623  4906   
Latvia               0      0    28    0     0     0     0   
Unknown              0      2     0    0     0     0     0   

century                                                      places  
region                                                               
Byzantine_Empire                                 [Byzantine_Empire]  
East Slavic       [Kiev, Moscow, Nizhny_Novgorod, NorthernRussia...  
Latvia                                                     [Latvia]  
Unknown                                                   [Unknown]  


In [8]:
# Count sum of verbs per Century (over all files)
df_century_value_count_after_filter = df.century.value_counts()
print(df_century_value_count_after_filter)

century
12    93208
15    42789
11    25932
17    25924
16    23880
13    20447
14     3095
Name: count, dtype: int64


In [9]:
# Sum of verbs by region and century
table_region_century_VERBS_ONLY = groupby_col_row_size_table_optional_filter(df, "region", "century", filter_col="POS" , filter_val="V-")
print(table_region_century_VERBS_ONLY)

century             11     12    13   14    15    16    17  \
region                                                       
Byzantine_Empire     0      0     0    0  1684     0     0   
East Slavic       5145  17992  3087  455  5730  3623  4906   
Latvia               0      0    28    0     0     0     0   
Unknown              0      2     0    0     0     0     0   

century                                                      places  
region                                                               
Byzantine_Empire                                 [Byzantine_Empire]  
East Slavic       [Kiev, Moscow, Nizhny_Novgorod, NorthernRussia...  
Latvia                                                     [Latvia]  
Unknown                                                   [Unknown]  


In [10]:
## Optional: Save results as tables -> comment out

In [11]:
#!pip install openpyxl

In [12]:
#table_place_century_VERBS_ONLY.to_excel("OUTPUTS/VERBS_by_PLACExCENTURY.xlsx")

In [13]:
#table_region_century_VERBS_ONLY.to_excel("OUTPUTS/VERBS_by_REGIONxCENTURY.xlsx")